# 04 — Model Comparison & Evaluation
**CFPB Consumer Complaint Classification** (Debt collection vs Credit card)

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Load cleaned data

In [ ]:
train_df = pd.read_csv('../data/cleaned_train.csv')
test_df = pd.read_csv('../data/cleaned_test.csv')

print(f'Train: {train_df.shape[0]} samples')
print(f'Test:  {test_df.shape[0]} samples')

## 2. TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

X_train = tfidf.fit_transform(train_df['Consumer complaint narrative'])
X_test = tfidf.transform(test_df['Consumer complaint narrative'])

y_train = train_df['Product'].values
y_test = test_df['Product'].values

label_map = {'Credit card': 0, 'Debt collection': 1}
y_train_bin = np.array([label_map[y] for y in y_train])
y_test_bin = np.array([label_map[y] for y in y_test])

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

## 3. Define models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes': MultinomialNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
}

## 4. Cross-Validation (5-Fold Stratified)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    acc_scores = cross_val_score(model, X_train, y_train_bin, cv=cv, scoring='accuracy')
    f1_scores = cross_val_score(model, X_train, y_train_bin, cv=cv, scoring='f1')
    cv_results[name] = {
        'accuracy_mean': acc_scores.mean(),
        'accuracy_std': acc_scores.std(),
        'f1_mean': f1_scores.mean(),
        'f1_std': f1_scores.std()
    }
    print(f'{name}: Accuracy={acc_scores.mean():.4f} F1={f1_scores.mean():.4f}')

## 5. CV score distribution plot

In [ ]:
cv_f1_all = {}
for name, model in models.items():
    cv_f1_all[name] = cross_val_score(model, X_train, y_train_bin, cv=cv, scoring='f1')

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#0f172a')

bp = ax.boxplot(cv_f1_all.values(), positions=range(1, len(cv_f1_all)+1), widths=0.5,
                patch_artist=True, notch=True,
                medianprops=dict(color='white', linewidth=2),
                whiskerprops=dict(color='#94a3b8'),
                capprops=dict(color='#94a3b8'),
                flierprops=dict(markerfacecolor='#f97316', marker='o', markersize=5))
colors = ['#3b82f6', '#10b981', '#f97316', '#a78bfa', '#f472b6', '#22d3ee']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_xticklabels(cv_f1_all.keys(), color='#e2e8f0', fontsize=11)
ax.set_ylabel('F1 Score', color='#e2e8f0', fontsize=12)
ax.set_title('Cross-Validation F1 Score Distribution', color='#f1f5f9', fontsize=14, fontweight='bold', pad=15)
ax.tick_params(colors='#94a3b8')
ax.spines['bottom'].set_color('#334155')
ax.spines['left'].set_color('#334155')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, color='#334155', alpha=0.5)
plt.tight_layout()
plt.savefig('../outputs/cv_f1_distribution.png', dpi=150, facecolor='#0f172a')
print('Saved cv_f1_distribution.png')

## 6. Hyperparameter Tuning

In [ ]:
param_grids = {
    'Logistic Regression': {'C': [0.1, 1, 10], 'penalty': ['l2'], 'solver': ['lbfgs']},
    'Naive Bayes': {'alpha': [0.01, 0.1, 0.5, 1.0]},
    'Decision Tree': {'max_depth': [10, 20, None], 'criterion': ['gini', 'entropy']},
    'Random Forest': {'n_estimators': [100, 200], 'max_depth': [10, None], 'min_samples_split': [2, 5]},
    'XGBoost': {'n_estimators': [100], 'max_depth': [3, 6], 'learning_rate': [0.1]},
    'LightGBM': {'n_estimators': [100], 'max_depth': [3, 6], 'learning_rate': [0.1], 'num_leaves': [31]}
}

tuned_models = {}
for name in models:
    print(f'Tuning {name}...')
    grid = GridSearchCV(models[name], param_grids[name], cv=cv, scoring='f1', verbose=0)
    grid.fit(X_train, y_train_bin)
    tuned_models[name] = grid.best_estimator_
    print(f'  Best params: {grid.best_params_}  F1={grid.best_score_:.4f}')

## 7. Evaluate on test set

In [ ]:
test_results = {}

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    test_results[name] = {
        'y_pred': y_pred,
        'y_prob': y_prob,
        'accuracy': accuracy_score(y_test_bin, y_pred),
        'precision': precision_score(y_test_bin, y_pred),
        'recall': recall_score(y_test_bin, y_pred),
        'f1': f1_score(y_test_bin, y_pred)
    }

    print(f'--- {name} ---')
    print(classification_report(y_test_bin, y_pred, target_names=['Credit card', 'Debt collection']))

## 8. Model comparison

In [ ]:
summary = pd.DataFrame({
    'Model': list(test_results.keys()),
    'Accuracy': [r['accuracy'] for r in test_results.values()],
    'Precision': [r['precision'] for r in test_results.values()],
    'Recall': [r['recall'] for r in test_results.values()],
    'F1 Score': [r['f1'] for r in test_results.values()]
})
summary = summary.sort_values('F1 Score', ascending=False).reset_index(drop=True)
print(summary.to_string(index=False))

best_model_name = summary.iloc[0]['Model']
print(f'\nBest model: {best_model_name} (F1 = {summary.iloc[0]["F1 Score"]:.4f})')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#0f172a')

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
x = np.arange(len(summary))
width = 0.2
metric_colors = ['#3b82f6', '#10b981', '#f97316', '#a78bfa']

for i, metric in enumerate(metrics_to_plot):
    bars = ax.bar(x + i * width, summary[metric], width, label=metric,
                  color=metric_colors[i], alpha=0.85)
    for bar, val in zip(bars, summary[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', color='#94a3b8', fontsize=8)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(summary['Model'], color='#e2e8f0', fontsize=11)
ax.set_ylabel('Score', color='#e2e8f0')
ax.set_title('Model Comparison', color='#f1f5f9', fontsize=14, fontweight='bold', pad=15)
ax.set_ylim(0.8, 1.02)
ax.legend(loc='lower right', facecolor='#1e293b', edgecolor='#334155', labelcolor='#e2e8f0')
ax.tick_params(colors='#94a3b8')
ax.spines['bottom'].set_color('#334155')
ax.spines['left'].set_color('#334155')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, color='#334155', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/model_comparison.png', dpi=150, facecolor='#0f172a')
print('Saved model_comparison.png')

## 9. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#0f172a')

roc_colors = ['#3b82f6', '#10b981', '#f97316', '#a78bfa', '#f472b6', '#22d3ee']
for i, (name, res) in enumerate(test_results.items()):
    fpr, tpr, _ = roc_curve(y_test_bin, res['y_prob'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=roc_colors[i], linewidth=2.5, label=f'{name} (AUC = {roc_auc:.4f})')

ax.plot([0, 1], [0, 1], color='#475569', linestyle='--', linewidth=1.5, alpha=0.7)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('False Positive Rate', color='#e2e8f0')
ax.set_ylabel('True Positive Rate', color='#e2e8f0')
ax.set_title('ROC Curves', color='#f1f5f9', fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='lower right', facecolor='#1e293b', edgecolor='#334155', labelcolor='#e2e8f0')
ax.tick_params(colors='#94a3b8')
ax.spines['bottom'].set_color('#334155')
ax.spines['left'].set_color('#334155')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/roc_curves.png', dpi=150, facecolor='#0f172a')
print('Saved roc_curves.png')

## 10. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.patch.set_facecolor('#0f172a')
fig.suptitle('Confusion Matrices', color='#f1f5f9', fontsize=16, fontweight='bold', y=1.01)

cm_colors = ['#3b82f6', '#10b981', '#f97316', '#a78bfa', '#f472b6', '#22d3ee']
for idx, (name, res) in enumerate(test_results.items()):
    ax = axes[idx // 3][idx % 3]
    ax.set_facecolor('#0f172a')
    cm = confusion_matrix(y_test_bin, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Credit card', 'Debt collection'],
                yticklabels=['Credit card', 'Debt collection'],
                ax=ax, annot_kws={'color': '#e2e8f0', 'fontsize': 14})
    ax.set_title(name, color=cm_colors[idx], fontsize=13, fontweight='bold')
    ax.set_ylabel('True Label', color='#94a3b8')
    ax.set_xlabel('Predicted Label', color='#94a3b8')
    ax.tick_params(colors='#94a3b8')

plt.tight_layout()
plt.savefig('../outputs/confusion_matrices.png', dpi=150, facecolor='#0f172a', bbox_inches='tight')
print('Saved confusion_matrices.png')

## 11. Feature Importance (Random Forest)

In [ ]:
rf_model = tuned_models['Random Forest']
feature_names = tfidf.get_feature_names_out()
importances = rf_model.feature_importances_
top_idx = np.argsort(importances)[-20:]

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#0f172a')
ax.barh(range(len(top_idx)), importances[top_idx], color='#a78bfa', alpha=0.85)
ax.set_yticks(range(len(top_idx)))
ax.set_yticklabels([feature_names[i] for i in top_idx], color='#e2e8f0', fontsize=11)
ax.set_xlabel('Importance', color='#e2e8f0')
ax.set_title('Top 20 Features — Random Forest', color='#f1f5f9', fontsize=14, fontweight='bold', pad=15)
ax.tick_params(colors='#94a3b8')
ax.spines['bottom'].set_color('#334155')
ax.spines['left'].set_color('#334155')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150, facecolor='#0f172a')
print('Saved feature_importance.png')

## 12. Save models and metrics

In [ ]:
for name, model in tuned_models.items():
    safe_name = name.lower().replace(' ', '_')
    joblib.dump(model, f'../outputs/{safe_name}_model.joblib')

joblib.dump(tfidf, '../outputs/tfidf_vectorizer.joblib')
print('All models saved')

In [ ]:
all_metrics = {}
for name, res in test_results.items():
    fpr, tpr, _ = roc_curve(y_test_bin, res['y_prob'])
    all_metrics[name] = {
        'accuracy': round(res['accuracy'], 4),
        'precision': round(res['precision'], 4),
        'recall': round(res['recall'], 4),
        'f1': round(res['f1'], 4),
        'auc': round(auc(fpr, tpr), 4)
    }

all_metrics['best_model'] = best_model_name

with open('../outputs/metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

print('Metrics saved')
print(json.dumps(all_metrics, indent=2))

In [ ]:
ml_preds_df = pd.DataFrame({
    'narrative': test_df['Consumer complaint narrative'].values,
    'true_label': y_test,
    'predicted_label': tuned_models[best_model_name].predict(X_test)
})
ml_preds_df.to_csv('../outputs/ml_predictions.csv', index=False)
print(f'Predictions saved from best model ({best_model_name})')